In [1]:
import re
import string
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
import nltk

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Download NLTK resources needed for tokenization, stop-word removal and lemmatization.
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("NLTK resources downloaded/available.")


In [ ]:
# Load the exact dataset specified in the assignment.
# The raw GitHub URL lets the notebook work without manually downloading the CSV first.

DATA_URL = (
    "https://raw.githubusercontent.com/WuraolaOyewusi/"
    "Health-Related-Headlines-Datasets-for-Natural-Language-Processing/"
    "master/Health_Related_Headlines_Dataset.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())


In [ ]:
# Basic dataset inspection
print("Missing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nCategory distribution:")
if "Category" in df.columns:
    display(df["Category"].value_counts())


## Task 1 — Data Preprocessing and Cleaning

The assignment requires:
- Tokenization
- Stop-word removal
- Lemmatization
- Lowercasing
- Punctuation removal
- Displaying cleaned text samples


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """Lowercase, remove punctuation/numbers, tokenize, remove stopwords and lemmatize."""
    if pd.isna(text):
        return ""

    # Lowercasing
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Keep alphabetic tokens only
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Tokenization
    tokens = word_tokenize(text)

    # Stop-word removal + lemmatization
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    return " ".join(cleaned_tokens)

# Use Headlines as the primary corpus.
df["Headlines"] = df["Headlines"].fillna("").astype(str)
df["Cleaned_Text"] = df["Headlines"].apply(clean_text)

# Remove empty rows after cleaning.
df = df[df["Cleaned_Text"].str.strip().ne("")].reset_index(drop=True)

print("Rows after cleaning:", len(df))
display(df[["Headlines", "Cleaned_Text"]].head(10))


### Preprocessing result

The `Cleaned_Text` column is now the NLP corpus. The transformation:
1. converts text to lowercase,
2. removes URLs and punctuation,
3. tokenizes the text,
4. removes common English stop words,
5. removes very short tokens,
6. lemmatizes the remaining words.


## Task 2 — Exploratory Text Analysis & Word Visualization

We will calculate:
- Word-frequency distribution
- Top 20 frequent words
- A word cloud of common health-related terms


In [ ]:
# Combine all cleaned documents into one token list.
all_tokens = " ".join(df["Cleaned_Text"]).split()

word_freq = Counter(all_tokens)

freq_df = pd.DataFrame(
    word_freq.most_common(20),
    columns=["Word", "Frequency"]
)

print("Top 20 most frequent words:")
display(freq_df)


In [ ]:
# Plot top 20 frequent words using matplotlib/seaborn.

plt.figure(figsize=(12, 7))
sns.barplot(
    data=freq_df,
    x="Frequency",
    y="Word"
)

plt.title("Top 20 Most Frequent Words in Health-Related Headlines")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()


In [ ]:
# Generate the word cloud.

wordcloud = WordCloud(
    width=1400,
    height=700,
    background_color="white",
    max_words=150,
    collocations=False
).generate(" ".join(all_tokens))

plt.figure(figsize=(16, 8))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Health-Related Terms", fontsize=18)
plt.show()


In [ ]:
# Word frequency distribution: rank vs frequency.

frequencies = sorted(word_freq.values(), reverse=True)

plt.figure(figsize=(12, 6))
plt.plot(
    range(1, len(frequencies) + 1),
    frequencies,
    linewidth=1
)

plt.title("Word Frequency Distribution")
plt.xlabel("Word Rank")
plt.ylabel("Frequency")
plt.yscale("log")
plt.xscale("log")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Task 3 — Topic Modelling using LDA

Latent Dirichlet Allocation (LDA) identifies groups of words that frequently occur together. Each topic is represented by its most important words.

For this assignment:
- Number of topics = **5**
- Top words displayed per topic = **5**


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Convert cleaned text into a document-term matrix.
vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=5,
    max_features=5000
)

dtm = vectorizer.fit_transform(df["Cleaned_Text"])

print("Document-Term Matrix shape:", dtm.shape)


In [ ]:
# Train the LDA model.

N_TOPICS = 5

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method="batch",
    max_iter=20
)

lda_model.fit(dtm)

print(f"LDA model trained with {N_TOPICS} topics.")


In [ ]:
# Display top 5 words for each topic.

feature_names = vectorizer.get_feature_names_out()

def display_topics(model, feature_names, n_top_words=5):
    topic_rows = []

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        topic_rows.append({
            "Topic": f"Topic {topic_idx + 1}",
            "Top 5 Words": ", ".join(top_words)
        })

    return pd.DataFrame(topic_rows)

topics_df = display_topics(
    lda_model,
    feature_names,
    n_top_words=5
)

display(topics_df)


### Interpretation of the topics

The exact words and their order are calculated when the notebook is executed on the dataset. In general, the topics represent recurring healthcare themes present in the headlines, such as disease/medical conditions, treatment or drugs, clinical research/trials, patient safety, and public-health related issues.

The topic labels should be assigned **after inspecting the five words produced by the LDA model**, rather than hard-coding labels before seeing the results.


In [ ]:
# Assign the dominant topic to every document.

document_topic_distribution = lda_model.transform(dtm)
df["Dominant_Topic"] = document_topic_distribution.argmax(axis=1) + 1

print("Number of documents assigned to each dominant topic:")
display(df["Dominant_Topic"].value_counts().sort_index())


In [ ]:
# Show representative headlines for each dominant topic.

for topic_num in range(1, N_TOPICS + 1):
    print(f"\n{'=' * 70}")
    print(f"TOPIC {topic_num} — REPRESENTATIVE HEADLINES")
    print(f"{'=' * 70}")

    sample_rows = df[df["Dominant_Topic"] == topic_num]["Headlines"].head(5)
    for headline in sample_rows:
        print("•", headline)


## LDA Visualization using pyLDAvis

pyLDAvis provides an interactive visualization of:
- the relative prevalence of topics,
- the words associated with each topic,
- the relevance of terms within each topic.

If the visualization does not render inside a particular Jupyter environment, the same analysis can still be submitted because the topic-word table above contains the required LDA results.


In [ ]:
import pyLDAvis
import pyLDAvis.lda_model

# Prepare the interactive LDA visualization.
pyLDAvis.enable_notebook()

lda_vis = pyLDAvis.lda_model.prepare(
    lda_model,
    dtm,
    vectorizer,
    mds="tsne"
)

pyLDAvis.display(lda_vis)


## Conclusion

### Task 1
The healthcare headline corpus was cleaned using lowercasing, punctuation removal, tokenization, stop-word removal, and lemmatization.

### Task 2
Word-frequency analysis and a word cloud were used to identify the most common terms appearing in the healthcare headlines.

### Task 3
LDA topic modeling was applied with five topics. The five most important words for each topic were extracted, and pyLDAvis was used to interactively explore topic prevalence and word relevance.

**Result:** NLP and topic modeling can reveal recurring themes and trends in large collections of healthcare-related text, making unstructured health information easier to explore and analyze.


### Submission note

Run all cells from top to bottom before submission so that the notebook contains:
- dataset preview,
- cleaned text samples,
- top-20 word frequency table,
- top-20 word chart,
- word cloud,
- frequency distribution,
- five LDA topics with top five words,
- representative headlines,
- interactive pyLDAvis output.
